# 5.7. Atskirų garso failų spektrogramų analizė

In [ ]:
import wave, csv
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from pathlib import Path

BASE_DIR     = Path('.')
SHOWCASE     = BASE_DIR / 'results' / 'audio_showcase' / 'exp_d_liepa_full'
AUD_EXAMPLES = BASE_DIR / 'results' / 'full_liepa_full_unetdilatedmaskcnn_v2' / 'audio_examples'
RESULTS_DIR  = BASE_DIR / 'results'
FIG_DIR      = BASE_DIR / 'figures'
FIG_DIR.mkdir(exist_ok=True)

EXP      = 'full_liepa_full_unetdilatedmaskcnn_v2'
EX1_ID   = 'ex02';  EX1_UTT = 'D197_S037_S197Vk_037_13_snr2p5'
EX2_ID   = 'ex05';  EX2_UTT = 'D200_S038_S200Vl_038_31_snr2p5'
EX3_STEM = 'worst_D522_S022_S522Mc_022_25_snr12p5'
EX3_UTT  = 'D522_S022_S522Mc_022_25_snr12p5'

N_FFT = 1024;  HOP = 256;  WIN = 1024
print('Konfiguracija uzkrauta.')

In [ ]:
def lt(v, dec=3):
    return f'{v:.{dec}f}'.replace('.', ',')

def read_wav(path):
    with wave.open(str(path), 'rb') as wf:
        nch = wf.getnchannels(); sr = wf.getframerate()
        raw = wf.readframes(wf.getnframes())
    d = np.frombuffer(raw, np.int16).astype(np.float32) / 32768.0
    if nch > 1: d = d.reshape(-1, nch)[:, 0]
    return d, sr

def compute_stft_db(signal):
    window = np.hanning(WIN)
    n_frames = 1 + (len(signal) - N_FFT) // HOP
    mag = np.zeros((N_FFT // 2 + 1, n_frames))
    for i in range(n_frames):
        seg = signal[i*HOP : i*HOP+N_FFT]
        if len(seg) < N_FFT: seg = np.pad(seg, (0, N_FFT - len(seg)))
        mag[:, i] = np.abs(np.fft.rfft(seg * window))
    return 20.0 * np.log10(mag + 1e-8)

_FLOAT_COLS = {
    'duration_sec','pesq_noisy','pesq_enhanced','delta_pesq',
    'stoi_noisy','stoi_enhanced','delta_stoi',
    'snr_db','snr_group','snr_noisy_est','snr_enhanced_est','delta_snr_est'
}

def load_metrics(utt_id):
    path = RESULTS_DIR / EXP / 'file_metrics.csv'
    with open(path, newline='', encoding='utf-8') as f:
        for row in csv.DictReader(f):
            if row['utt_id'] == utt_id:
                return {k: float(v) if k in _FLOAT_COLS else v for k, v in row.items()}
    raise KeyError(utt_id)

def make_spectrogram_fig(wav_dir, stem, title):
    sc, sr = read_wav(wav_dir / f'{stem}_clean.wav')
    sn, _  = read_wav(wav_dir / f'{stem}_noisy.wav')
    se, _  = read_wav(wav_dir / f'{stem}_enhanced.wav')
    dur = len(sc) / sr;  freq_max = sr / 2
    S = [compute_stft_db(x) for x in [sc, sn, se]]
    vmin = min(x.min() for x in S);  vmax = max(x.max() for x in S)
    fig, axes = plt.subplots(1, 3, figsize=(12, 3.8))
    fig.subplots_adjust(left=0.07, right=0.91, bottom=0.14, top=0.88, wspace=0.28)
    ims = []
    titles_lt = ['Svarus garsas', 'Triuksmingas garsas', 'Isvalytas garsas']
    for ax, spec, t in zip(axes, S, titles_lt):
        im = ax.imshow(spec, aspect='auto', origin='lower',
                       extent=[0, dur, 0, freq_max], vmin=vmin, vmax=vmax, cmap='magma')
        ims.append(im)
        ax.set_title(t, fontsize=10, pad=4)
        ax.set_xlabel('Laikas (s)', fontsize=9)
        ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{v:.1f}'.replace('.', ',')))
        ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{int(v):,}'.replace(',', ' ')))
    axes[0].set_ylabel('Daznis (Hz)', fontsize=9)
    cbar = fig.colorbar(ims[0], ax=axes, location='right', shrink=0.85, pad=0.01)
    cbar.set_label('Amplitude (dB)', fontsize=9)
    cbar.ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{v:.0f}'.replace('.', ',')))
    if title: fig.suptitle(title, fontsize=10, y=0.99)
    return fig, dur

print('Pagalbines funkcijos apibreztos.')

In [ ]:
m1 = load_metrics(EX1_UTT)
m2 = load_metrics(EX2_UTT)
m3 = load_metrics(EX3_UTT)
for label, m in [('1 TMETRO', m1), ('2 DLIVING', m2), ('3 12,5dB', m3)]:
    print(f'Pavyzdys {label}: PESQ {lt(m["pesq_noisy"],3)}->{lt(m["pesq_enhanced"],3)} '
          f'STOI {lt(m["stoi_noisy"],4)}->{lt(m["stoi_enhanced"],4)} '
          f'DSNR={lt(m["delta_snr_est"],2)}')

In [ ]:
fig32, dur1 = make_spectrogram_fig(SHOWCASE, EX1_ID,
    f'SNR = {lt(m1["snr_db"],1)} dB, triuksmas: {m1["noise_type"]} (pilnas LIEPA modelis)')
fig32.savefig(FIG_DIR / '32_spektrogramos_2_5_snr.png', dpi=150, bbox_inches='tight')
fig32.savefig(FIG_DIR / '32_spektrogramos_2_5_snr.svg', bbox_inches='tight')
plt.show()
print(f'32 pav. issaugotas.')

SNR = 2,5 dB, triukšmas TMETRO.



In [ ]:
fig33, dur2 = make_spectrogram_fig(SHOWCASE, EX2_ID,
    f'SNR = {lt(m2["snr_db"],1)} dB, triuksmas: {m2["noise_type"]} (pilnas LIEPA modelis)')
fig33.savefig(FIG_DIR / '33_spektrogramos_2_5_snr.png', dpi=150, bbox_inches='tight')
fig33.savefig(FIG_DIR / '33_spektrogramos_2_5_snr.svg', bbox_inches='tight')
plt.show()
print(f'33 pav. issaugotas.')

SNR = 2,5 dB, triukšmas DLIVING



In [ ]:
fig34s, dur3 = make_spectrogram_fig(AUD_EXAMPLES, EX3_STEM,
    f'SNR = {lt(m3["snr_db"],1)} dB, triuksmas: {m3["noise_type"]} (pilnas LIEPA modelis - ribinis atvejis)')
fig34s.savefig(FIG_DIR / '34_spektrogramos_12_5_snr.png', dpi=150, bbox_inches='tight')
fig34s.savefig(FIG_DIR / '34_spektrogramos_12_5_snr.svg', bbox_inches='tight')
plt.show()
print(f'34 pav. issaugotas.')

SNR = 12,5 dB, triukšmas TMETRO



In [ ]:
EXAMPLES_M = [('1 pav.', SHOWCASE, EX1_ID, EX1_UTT),
              ('2 pav.', SHOWCASE, EX2_ID, EX2_UTT)]

def snr_from_audio(clean, noisy):
    noise = noisy - clean
    return 10 * np.log10(np.sum(clean**2) / (np.sum(noise**2) + 1e-12))

rows_data = []
for label, wav_dir, ex_id, utt_id in EXAMPLES_M:
    sc, sr_ = read_wav(wav_dir / f'{ex_id}_clean.wav')
    sn, _   = read_wav(wav_dir / f'{ex_id}_noisy.wav')
    se, _   = read_wav(wav_dir / f'{ex_id}_enhanced.wav')
    m_      = load_metrics(utt_id)
    rows_data.append({'label': label, 'noise': m_['noise_type'], 'file': ex_id,
        'dur': len(sc)/sr_,
        'snr_n': snr_from_audio(sc, sn), 'snr_e': snr_from_audio(sc, se),
        'd_snr': snr_from_audio(sc, se) - snr_from_audio(sc, sn),
        'pesq_n': m_['pesq_noisy'], 'pesq_e': m_['pesq_enhanced'], 'd_pesq': m_['delta_pesq'],
        'stoi_n': m_['stoi_noisy'], 'stoi_e': m_['stoi_enhanced'], 'd_stoi': m_['delta_stoi']})

HEADER_M = ['Pavyzdys','Triuksmo tipas','Failo pavadinimas','Trukme, s',
            'SNR pries, dB','SNR po, dB','DSNR, dB',
            'PESQ pries','PESQ po','DPESQ','STOI pries','STOI po','DSTOI']

def fmt_row(r):
    d4 = 4 if abs(r['d_stoi']) < 0.02 else 3
    return [r['label'], r['noise'], r['file'], lt(r['dur'],2),
            lt(r['snr_n'],2), lt(r['snr_e'],2), lt(r['d_snr'],2),
            lt(r['pesq_n'],3), lt(r['pesq_e'],3), lt(r['d_pesq'],3),
            lt(r['stoi_n'],3), lt(r['stoi_e'],3), lt(r['d_stoi'],d4)]

csv_m = FIG_DIR / 'atskiru_failu_metrikos.csv'
with open(csv_m, 'w', newline='', encoding='utf-8-sig') as f:
    csv.writer(f, delimiter=';').writerows([HEADER_M]+[fmt_row(r) for r in rows_data])
print(f'CSV: {csv_m}')
print()
print('\t'.join(HEADER_M))
print('\t'.join(['---']*len(HEADER_M)))
for r in rows_data: print('\t'.join(fmt_row(r)))

---
## Laiko srities signalų palyginimas

Tie patys signalai kaip spektrogramose, vaizduojami laiko srityje. Visi trys vieno pavyzdžio paneliai naudoja **bendrą amplitudės skalę**.

In [ ]:
BLUE = '#1a4d8f'

def plot_time_domain(wav_dir, stem, title, fbase):
    sc, sr = read_wav(wav_dir / f'{stem}_clean.wav')
    sn, _  = read_wav(wav_dir / f'{stem}_noisy.wav')
    se, _  = read_wav(wav_dir / f'{stem}_enhanced.wav')
    t    = np.arange(len(sc)) / sr
    peak = max(np.abs(sc).max(), np.abs(sn).max(), np.abs(se).max())
    ylim = (-peak * 1.08, peak * 1.08)

    fig, axes = plt.subplots(3, 1, figsize=(10, 6), sharex=True)
    fig.subplots_adjust(left=0.10, right=0.98, bottom=0.10, top=0.92, hspace=0.38)

    panel_labels = ['Švarus signalas', 'Triukšmingas signalas', 'Apdorotas signalas']

    # Downsample for display: keep ~4000 pts/s so oscillations are visible
    step = max(1, sr // 4000)
    t_d  = t[::step]

    for i, (ax, sig, lbl) in enumerate(zip(axes, [sc, sn, se], panel_labels)):
        sig_d = sig[::step]
        ax.plot(t_d, sig_d, color=BLUE, linewidth=0.6)
        ax.set_title(lbl, fontsize=10, pad=3)
        ax.set_ylabel('Amplitudė', fontsize=9)
        ax.set_ylim(ylim)
        ax.axhline(0, color='#aaa', linewidth=0.5, linestyle='--')
        ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{v:.1f}'.replace('.', ',')))
        ax.tick_params(labelsize=8)
        for sp in ['top', 'right']: ax.spines[sp].set_visible(False)

    axes[-1].set_xlabel('Laikas, s', fontsize=9)
    axes[-1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{v:.1f}'.replace('.', ',')))

    fig.suptitle(title, fontsize=10, y=0.98)
    fig.savefig(FIG_DIR / f'{fbase}.png', dpi=300, bbox_inches='tight')
    fig.savefig(FIG_DIR / f'{fbase}.svg', bbox_inches='tight')
    plt.show()
    print(f'{fbase} išsaugotas.')

plot_time_domain(SHOWCASE, EX1_ID,
    '2,5 dB SNR TMETRO pavyzdžio signalai laiko srityje',
    '34_laiko_sritis_tmetro_2_5_snr')

plot_time_domain(SHOWCASE, EX2_ID,
    '2,5 dB SNR DLIVING pavyzdžio signalai laiko srityje',
    '35_laiko_sritis_dliving_2_5_snr')

---
## Garso failų eksportavimas ir perklausymas

Išsaugomi atskiri ir sujungti palyginimo `.wav` failai. Sujungtuose failuose seka: **švarus → 1 s tylos → triukšmingas → 1 s tylos → išvalytas**.

In [ ]:
AUDIO_OUT = FIG_DIR / 'audio'
AUDIO_OUT.mkdir(exist_ok=True)
SR_OUT  = 16000
SILENCE = np.zeros(SR_OUT, dtype=np.float32)

def save_wav(path, signal, sr=SR_OUT):
    peak = np.abs(signal).max()
    if peak > 1.0: signal = signal / peak
    pcm = np.clip(signal * 32767, -32768, 32767).astype(np.int16)
    with wave.open(str(path), 'wb') as wf:
        wf.setnchannels(1); wf.setsampwidth(2); wf.setframerate(sr)
        wf.writeframes(pcm.tobytes())

def make_comparison(c, n, e):
    peak = max(np.abs(c).max(), np.abs(n).max(), np.abs(e).max())
    scale = 1.0 / peak if peak > 1.0 else 1.0
    return np.concatenate([c*scale, SILENCE, n*scale, SILENCE, e*scale])

def wav_dur(path):
    with wave.open(str(path), 'rb') as wf: return wf.getnframes() / wf.getframerate()

tmetro_c, _ = read_wav(SHOWCASE / f'{EX1_ID}_clean.wav')
tmetro_n, _ = read_wav(SHOWCASE / f'{EX1_ID}_noisy.wav')
tmetro_e, _ = read_wav(SHOWCASE / f'{EX1_ID}_enhanced.wav')
dliving_c, _ = read_wav(SHOWCASE / f'{EX2_ID}_clean.wav')
dliving_n, _ = read_wav(SHOWCASE / f'{EX2_ID}_noisy.wav')
dliving_e, _ = read_wav(SHOWCASE / f'{EX2_ID}_enhanced.wav')

AUDIO_FILES_SAVE = [
    ('tmetro_clean.wav',                            tmetro_c),
    ('tmetro_noisy_2_5db.wav',                      tmetro_n),
    ('tmetro_enhanced.wav',                         tmetro_e),
    ('tmetro_comparison_clean_noisy_enhanced.wav',  make_comparison(tmetro_c, tmetro_n, tmetro_e)),
    ('dliving_clean.wav',                           dliving_c),
    ('dliving_noisy_2_5db.wav',                     dliving_n),
    ('dliving_enhanced.wav',                        dliving_e),
    ('dliving_comparison_clean_noisy_enhanced.wav', make_comparison(dliving_c, dliving_n, dliving_e)),
]
for fname, sig in AUDIO_FILES_SAVE:
    p = AUDIO_OUT / fname
    save_wav(p, sig)
    print(f'  {fname:55s}  {lt(wav_dur(p),2)} s')

AUDIO_META = [
    ('1','TMETRO', 'Svarus',      'figures/audio/tmetro_clean.wav'),
    ('1','TMETRO', 'Triuksmingas','figures/audio/tmetro_noisy_2_5db.wav'),
    ('1','TMETRO', 'Isvalytas',   'figures/audio/tmetro_enhanced.wav'),
    ('1','TMETRO', 'Palyginimas', 'figures/audio/tmetro_comparison_clean_noisy_enhanced.wav'),
    ('2','DLIVING','Svarus',      'figures/audio/dliving_clean.wav'),
    ('2','DLIVING','Triuksmingas','figures/audio/dliving_noisy_2_5db.wav'),
    ('2','DLIVING','Isvalytas',   'figures/audio/dliving_enhanced.wav'),
    ('2','DLIVING','Palyginimas', 'figures/audio/dliving_comparison_clean_noisy_enhanced.wav'),
]
HDR_A = ['Pavyzdys','Triuksmo tipas','Signalo tipas','Failo kelias','Trukme, s','Diskretizavimo daznis']
csv_a = FIG_DIR / 'garso_pavyzdziu_failai.csv'
with open(csv_a, 'w', newline='', encoding='utf-8-sig') as f:
    w = csv.writer(f, delimiter=';'); w.writerow(HDR_A)
    for pav, noise, sig, rel in AUDIO_META:
        w.writerow([pav, noise, sig, rel, lt(wav_dur(BASE_DIR/rel),2), '16 000 Hz'])
print(f'\nCSV: {csv_a}')

### TMETRO pavyzdys – SNR = 2,5 dB

In [ ]:
from IPython.display import Audio, display

print('TMETRO pavyzdys - svarus signalas')
display(Audio(filename=str(AUDIO_OUT / 'tmetro_clean.wav')))
print('TMETRO pavyzdys - triuksmingas signalas (SNR = 2,5 dB)')
display(Audio(filename=str(AUDIO_OUT / 'tmetro_noisy_2_5db.wav')))
print('TMETRO pavyzdys - modelio apdorotas signalas')
display(Audio(filename=str(AUDIO_OUT / 'tmetro_enhanced.wav')))
print('TMETRO pavyzdys - palyginimas: svarus -> triuksmingas -> isvalytas')
display(Audio(filename=str(AUDIO_OUT / 'tmetro_comparison_clean_noisy_enhanced.wav')))

### DLIVING pavyzdys – SNR = 2,5 dB

In [ ]:
print('DLIVING pavyzdys - svarus signalas')
display(Audio(filename=str(AUDIO_OUT / 'dliving_clean.wav')))
print('DLIVING pavyzdys - triuksmingas signalas (SNR = 2,5 dB)')
display(Audio(filename=str(AUDIO_OUT / 'dliving_noisy_2_5db.wav')))
print('DLIVING pavyzdys - modelio apdorotas signalas')
display(Audio(filename=str(AUDIO_OUT / 'dliving_enhanced.wav')))
print('DLIVING pavyzdys - palyginimas: svarus -> triuksmingas -> isvalytas')
display(Audio(filename=str(AUDIO_OUT / 'dliving_comparison_clean_noisy_enhanced.wav')))

In [ ]:
print('=' * 62)
print('FAILU SUVESTINE')
print('=' * 62)
print(f'Katalogas: {AUDIO_OUT}')
print()
print('Atskiri failai:')
for pav, noise, sig, rel in AUDIO_META:
    if sig != 'Palyginimas':
        print(f'  [{pav}] {noise:8s} {sig:14s}  {rel.split("/")[-1]}')
print()
print('Rekomenduojami GYNIMO SKAIDREMS:')
for pav, noise, sig, rel in AUDIO_META:
    if sig == 'Palyginimas':
        fname = rel.split('/')[-1]
        print(f'  STAR  {fname}  ({lt(wav_dur(BASE_DIR/rel),2)} s)')
        print(f'        svarus -> 1s tylos -> triuksmingas -> 1s tylos -> isvalytas')